In [1]:
import cv2
import numpy as np
import sys
import math

np.set_printoptions(threshold=sys.maxsize)

# Importing and Preprocessing Any Map

In [2]:
_map = cv2.imread("map_1764936172.pgm")
print(_map.shape)
cv2.imshow("test", _map)
cv2.waitKey()
cv2.destroyAllWindows()

(474, 124, 3)


In [10]:
_map = cv2.imread("map_1764936172.pgm")
_map = cv2.cvtColor(_map, cv2.COLOR_BGR2GRAY)
_, thr = cv2.threshold(_map, 250, 255, cv2.THRESH_BINARY)

# Operate a morphology 
ellipse_kernel_1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

_clean_map = cv2.morphologyEx(thr, cv2.MORPH_OPEN, ellipse_kernel_1)
_clean_map = cv2.erode(_clean_map, ellipse_kernel_1, iterations=5)

# Find the largest connected area
num_labels, labels, stats, centroid = cv2.connectedComponentsWithStats(thr)

_max_i = 0
_max_area = 0
for i in range(1, num_labels):
    _area = stats[i, cv2.CC_STAT_AREA]
    if _area > _max_area:
        _max_i = i
        _max_area = _area

mask = (labels == _max_i).astype(np.uint8) * 255
_filtered_map = cv2.bitwise_and(thr, thr, mask=mask)

cv2.imshow("test", _filtered_map)
cv2.waitKey()
cv2.destroyAllWindows()
cv2.imwrite("map_compete_bw.png", _filtered_map)

True

In [19]:
_map = cv2.imread("map_compete_bw_ann.png")
_map = cv2.cvtColor(_map, cv2.COLOR_BGR2GRAY)

# Operate a morphology 
ellipse_kernel_1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

_clean_map = cv2.erode(_map, ellipse_kernel_1, iterations=3)

# Find the skeleton
skeleton = cv2.ximgproc.thinning(_clean_map, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)

cv2.imshow("test", _clean_map)
cv2.waitKey()
cv2.destroyAllWindows()

In [3]:
_map = cv2.imread("map_1764936172.pgm")
_map = cv2.cvtColor(_map, cv2.COLOR_BGR2GRAY)
_, thr = cv2.threshold(_map, 250, 255, cv2.THRESH_BINARY)

# Find the largest connected area
num_labels, labels, stats, centroid = cv2.connectedComponentsWithStats(thr)

_max_i = 0
_max_area = 0
for i in range(1, num_labels):
    _area = stats[i, cv2.CC_STAT_AREA]
    if _area > _max_area:
        _max_i = i
        _max_area = _area

mask = (labels == _max_i).astype(np.uint8) * 255
_filtered_map = cv2.bitwise_and(thr, thr, mask=mask)

# Operate a morphology 
ellipse_kernel_1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2,2))

_clean_map = cv2.morphologyEx(_filtered_map, cv2.MORPH_OPEN, ellipse_kernel_1)
_clean_map = cv2.dilate(_clean_map, ellipse_kernel_1, iterations=1)

# Find the skeleton
skeleton = cv2.ximgproc.thinning(_clean_map, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)

cv2.imshow("test", skeleton)
cv2.waitKey()
cv2.destroyAllWindows()

In [51]:
_map = cv2.imread("image0.jpg")
_map = cv2.cvtColor(_map, cv2.COLOR_BGR2GRAY)

# Invert the image
_, thr = cv2.threshold(_map, 220, 255, cv2.THRESH_BINARY_INV)

# Find the largest connected area
num_labels, labels, stats, centroid = cv2.connectedComponentsWithStats(thr)

_max_i = 0
_max_area = 0
for i in range(1, num_labels):
    _area = stats[i, cv2.CC_STAT_AREA]
    if _area > _max_area:
        _max_i = i
        _max_area = _area

mask = (labels == _max_i).astype(np.uint8) * 255
_filtered_map = cv2.bitwise_and(thr, thr, mask=mask)

# Find the skeleton
skeleton = cv2.ximgproc.thinning(_filtered_map, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)

cv2.imshow("test", skeleton)
cv2.waitKey()
cv2.destroyAllWindows()

# Building the Graph

In [52]:
import sknw
import networkx as nx

graph = sknw.build_sknw((skeleton > 0).astype(np.uint8), multi=False)
# graph.nodes = skeleton points
# graph.edges = paths along skeleton

# Example: get main path
edges = list(graph.edges(data=True))
print(edges)

[(0, 1, {'pts': array([[60, 69],
       [58, 71],
       [58, 72],
       [58, 73],
       [58, 74],
       [59, 75],
       [59, 76],
       [60, 77],
       [62, 79]], dtype=int16), 'weight': 12.485281374238571}), (0, 4, {'pts': array([[60, 69],
       [62, 68],
       [63, 67],
       [64, 67],
       [65, 66],
       [66, 66],
       [67, 65],
       [68, 65],
       [69, 65],
       [70, 64],
       [71, 64],
       [72, 64],
       [73, 63],
       [74, 63],
       [75, 63],
       [76, 63],
       [77, 62],
       [78, 62],
       [79, 62],
       [81, 62]], dtype=int16), 'weight': 23.72134935173836}), (1, 2, {'pts': array([[62, 79],
       [63, 81],
       [64, 82]], dtype=int16), 'weight': 3.6502815398728847}), (2, 2, {'pts': array([[64, 82],
       [64, 83],
       [64, 82]], dtype=int16), 'weight': 2.0}), (2, 3, {'pts': array([[64, 82],
       [66, 84],
       [67, 84],
       [68, 85],
       [69, 85],
       [70, 86],
       [71, 86],
       [72, 87],
       [73, 87],
    

In [58]:
# Test and visualize if the path is correctly coppied
path_test = np.zeros_like(_map)
path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for _edge in edges:
    path_test = cv2.circle(path_test, (_edge[2]['pts'][0][1], _edge[2]['pts'][0][0]), 1, (0, 0, 255), 2)
    path_test = cv2.circle(path_test, (_edge[2]['pts'][-1][1], _edge[2]['pts'][-1][0]), 1, (0, 0, 255), 2)
    path_test = cv2.putText(path_test, f"{_edge[0]}", (_edge[2]['pts'][0][1], _edge[2]['pts'][0][0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    for _coor in _edge[2]['pts']:
        path_test[_coor[0], _coor[1]] = (255, 255, 255)

cv2.imshow("test", path_test)
cv2.waitKey()
cv2.destroyAllWindows()

# Creating The Path

In [109]:
# Step 1: Relabel nodes consistently
sorted_nodes = sorted(graph.nodes())
mapping = {old:new for new,old in enumerate(sorted_nodes)}
graph = nx.relabel_nodes(graph, mapping, copy=True)

subgraph = graph.subgraph(sorted_nodes).copy()

# Prepare visited
node_list = list(subgraph.nodes())
idx = {n: i for i, n in enumerate(node_list)}
visited = [False] * len(node_list)

_path = []
strt = 7
cur = strt

while True:
    if all(visited):
        break

    visited[idx[cur]] = True

    for nbr in subgraph.neighbors(cur):
        if not visited[idx[nbr]]:
            _path.append((cur, nbr))
            cur = nbr
            break

# Connect the last path
_path.append((cur, strt))

# Concat all of the intermidiate nodes
_all_nodes = []
for p in _path:
    _all_nodes.extend(graph[p[0]][p[1]]['pts'])

# Convert to numpy array
_all_nodes = np.array(_all_nodes)

print(_path)
print(_all_nodes)


[(7, 6), (6, 5), (5, 3), (3, 2), (2, 1), (1, 0), (0, 4), (4, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 14), (14, 16), (16, 17), (17, 20), (20, 22), (22, 21), (21, 19), (19, 18), (18, 15), (15, 13), (13, 7)]
[[113 104]
 [115 105]
 [116 105]
 [118 106]
 [ 87  95]
 [ 90  96]
 [ 91  97]
 [ 92  97]
 [ 93  98]
 [ 94  98]
 [ 95  98]
 [ 96  99]
 [ 97  99]
 [ 98  99]
 [ 99 100]
 [100 100]
 [101 101]
 [102 101]
 [103 101]
 [104 102]
 [105 102]
 [106 102]
 [107 102]
 [108 102]
 [109 102]
 [110 103]
 [111 103]
 [113 104]
 [ 78  90]
 [ 81  92]
 [ 82  92]
 [ 83  93]
 [ 84  94]
 [ 87  95]
 [ 64  82]
 [ 66  84]
 [ 67  84]
 [ 68  85]
 [ 69  85]
 [ 70  86]
 [ 71  86]
 [ 72  87]
 [ 73  87]
 [ 74  88]
 [ 75  88]
 [ 76  89]
 [ 78  90]
 [ 62  79]
 [ 63  81]
 [ 64  82]
 [ 60  69]
 [ 58  71]
 [ 58  72]
 [ 58  73]
 [ 58  74]
 [ 59  75]
 [ 59  76]
 [ 60  77]
 [ 62  79]
 [ 60  69]
 [ 62  68]
 [ 63  67]
 [ 64  67]
 [ 65  66]
 [ 66  66]
 [ 67  65]
 [ 68  65]
 [ 69  65]
 [ 70  64]
 [ 71  64]
 [ 72  64]
 [ 73  6

In [103]:
_all_nodes = _all_nodes[::-1]

In [7]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((600, 600, 3))
print(path_test.shape)
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(_all_nodes.shape[0]):
    path_test[_all_nodes[i][0] + 200, _all_nodes[i][1] + 200] = (255, 255, 255)

cv2.imshow("test", path_test)
cv2.waitKey()
cv2.destroyAllWindows()

(600, 600, 3)


In [22]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((1200, 1200, 3))
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(_all_nodes.shape[0]):
    n_1 = _all_nodes[i]
    n_2 = _all_nodes[(i+1)%_all_nodes.shape[0]]
    path_test = cv2.arrowedLine(path_test, (n_1[1]+200, n_1[0]+200), (n_2[1]+200, n_2[0]+200), (0, 255, 0), 1, cv2.LINE_AA, tipLength=0.5)

cv2.imshow("test", path_test)
cv2.waitKey()
cv2.destroyAllWindows()

In [111]:
# Test and visualized if the node is correctly structured
path_test = np.zeros((1200, 1200, 3))
#path_test = cv2.cvtColor(path_test, cv2.COLOR_GRAY2BGR)
for i in range(_all_nodes.shape[0]):
    n_1 = _all_nodes[i]
    n_2 = _all_nodes[(i+1)%_all_nodes.shape[0]]
    path_test = cv2.arrowedLine(path_test, (n_1[1]+200, n_1[0]+200), (n_2[1]+200, n_2[0]+200), (0, 255, 0), 1, cv2.LINE_AA, tipLength=0.5)
    cv2.imshow("test", path_test)
    c = cv2.waitKey(10)
    if c == 27:
        break
cv2.destroyAllWindows()

In [11]:
import yaml
with open('map_1764687152.yaml', 'r') as f:
    data = yaml.load(f, Loader=yaml.SafeLoader)
print(data)

_res = data["resolution"]
_origin = data["origin"]

{'image': 'map_1764687152.pgm', 'mode': 'trinary', 'resolution': 0.05, 'origin': [-4.29, -8.61, 0], 'negate': 0, 'occupied_thresh': 0.65, 'free_thresh': 0.25}


In [37]:
# Convert the pixel coordinate to map coordinate
_world_coord = []
for i in range(_all_nodes.shape[0]):
    n_1 = _all_nodes[i]
    n_2 = _all_nodes[(i+1)%_all_nodes.shape[0]]
    _coor_x_1 = (_res * n_1[1]) + _origin[0]
    _coor_y_1 = (_res * (_map.shape[0] - n_1[0])) + _origin[1]
    _coor_x_2 = (_res * n_2[1]) + _origin[0]
    _coor_y_2 = (_res * (_map.shape[0] - n_2[0])) + _origin[1]
    _yaw = math.atan2(_coor_y_2-_coor_y_1, _coor_x_2-_coor_x_1)
    _world_coord.append((_coor_x_1, _coor_y_1, _yaw))
print(_world_coord)

[(-0.43999999999999995, 5.140000000000001, 2.3561944901923417), (-0.6399999999999997, 5.340000000000002, 3.141592653589793), (-0.69, 5.340000000000002, 2.3561944901923537), (-0.7399999999999998, 5.390000000000001, 3.141592653589793), (-0.79, 5.390000000000001, 3.141592653589793), (-0.8399999999999999, 5.390000000000001, 3.141592653589793), (-0.8899999999999997, 5.390000000000001, 2.3561944901923404), (-0.94, 5.440000000000001, 3.141592653589793), (-0.9899999999999998, 5.440000000000001, 3.141592653589793), (-1.04, 5.440000000000001, 3.141592653589793), (-1.0899999999999999, 5.440000000000001, 3.141592653589793), (-1.1399999999999997, 5.440000000000001, 2.3561944901923404), (-1.19, 5.490000000000002, 3.141592653589793), (-1.2399999999999998, 5.490000000000002, 3.141592653589793), (-1.29, 5.490000000000002, 3.141592653589793), (-1.3399999999999999, 5.490000000000002, 3.141592653589793), (-1.3899999999999997, 5.490000000000002, 3.141592653589793), (-1.44, 5.490000000000002, 2.356194490192

In [38]:
import csv
with open('path.csv', "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["x", "y", "yaw"])
    writer.writerows(_world_coord)
